# Day 02 – Clustering, Annotation & Light Data Integration

We turn the UMAP blobs into real cell types and show a bite-sized batch-correction trick so multiple datasets line up.


### What happens today?
1. Load the cleaned data from Day 01 (or quickly rebuild it if missing).
2. Cluster with Leiden and assign cell-type names using a tiny marker dictionary.
3. Duplicate the data to mimic a second batch and run a simple Combat-style correction.


In [ ]:
# Install the needed libraries once (delete the # to run)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


### Step 1 – Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print('⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.')

try:
    import gseapy as gp
except ImportError:
    gp = None
    print('⚠️ Install gseapy (pip install gseapy) to run the GSEA step.')

try:
    import scvelo as scv
except ImportError:
    scv = None
    print('⚠️ Install scvelo (pip install scvelo) to run the RNA velocity step.')

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


### Step 2 – Grab the cleaned matrix from Day 01

In [ ]:
SHARED_DIR = Path('..') / 'shared_data'
SHARED_DIR.mkdir(parents=True, exist_ok=True)
day1_file = SHARED_DIR / 'day01_preprocessed.h5ad'

print('Looking for Day 01 data at', day1_file)

def build_day1_from_scratch():
    """Run the Day 01 cleaning steps so later lessons still work."""
    data = sc.datasets.pbmc3k()
    data.var_names_make_unique()
    data.layers['counts'] = data.X.copy()
    data.obs['source_dataset'] = 'pbmc3k'
    data.raw = data

    sc.pp.filter_cells(data, min_genes=200)
    sc.pp.filter_genes(data, min_cells=3)
    data.var['mt'] = data.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], inplace=True)
    data = data[data.obs['pct_counts_mt'] < 15, :]

    sc.pp.normalize_total(data, target_sum=1e4)
    sc.pp.log1p(data)
    sc.pp.highly_variable_genes(data, n_top_genes=2000, subset=True)
    sc.pp.scale(data, max_value=10)

    sc.tl.pca(data, n_comps=50)
    sc.pp.neighbors(data, n_neighbors=15)
    sc.tl.umap(data)
    return data

if day1_file.exists():
    adata_day2 = sc.read(day1_file)
    print('✅ Loaded preprocessed data from Day 01 file.')
else:
    print('⚠️ Day 01 file not found. Re-running a mini Day 01 pipeline now...')
    adata_day2 = build_day1_from_scratch()


### Step 3 – Cluster and tag each cluster with a friendly name

In [ ]:
sc.tl.leiden(adata_day2, resolution=0.5, key_added='leiden')
marker_map = {
    '0': 'Naive T',
    '1': 'Memory T',
    '2': 'B cell',
    '3': 'NK',
    '4': 'Myeloid',
    '5': 'Plasma',
}
adata_day2.obs['cell_type'] = adata_day2.obs['leiden'].map(marker_map).fillna('Other')
sc.pl.umap(adata_day2, color=['leiden', 'cell_type'], frameon=False)

adata_day2.obs['batch'] = 'Batch_A'


### Step 4 – Fake a second batch and apply a Combat correction

In [ ]:
adata_batch_b = adata_day2[:1000].copy()
adata_batch_b.obs['batch'] = 'Batch_B'

adata_integrated = sc.concat([adata_day2, adata_batch_b], join='outer')
sc.pp.normalize_total(adata_integrated, target_sum=1e4)
sc.pp.log1p(adata_integrated)
sc.pp.highly_variable_genes(adata_integrated, n_top_genes=2000, subset=True)
sc.pp.scale(adata_integrated, max_value=10)
sc.tl.pca(adata_integrated)
sc.pp.neighbors(adata_integrated, n_neighbors=15)
sc.tl.umap(adata_integrated)
sc.pp.combat(adata_integrated, key='batch')

sc.pl.umap(adata_integrated, color=['batch'], title=['After simple integration'], frameon=False)


### Step 5 – Save annotated data for future days

In [ ]:
output_path = SHARED_DIR / 'day02_annotated.h5ad'
adata_day2.write(output_path)
print(f'Saved annotated data to {output_path}')
